# Lineament Clustering Tool  
### *(Associated with the following publication)*

**Citation:**  
Hiroaki Koge, Taichi Sato, Jun Arimoto, Makoto Otsubo, Saki Ishino, Yoshiaki Suzuki, Osamu Ishizuka, Yumiko Harigane, Ayanori Misawa, Takahiko Inoue, Mikiya Yamashita, Seishiro Furuyama, Yuka Yokoyama, Yusuke Sato, Takami Mori, Hiroki Minami, Chiori Tamura;  
**Inception of ridge–ridge–ridge triple junctions: Morphostructural analysis and dynamics in the early back-arc extension of the northern Okinawa Trough**.  *Geology*, 2025, 53 (3): 269–273. https://doi.org/10.1130/G52640.1

---

## About this notebook

This IPython Notebook contains the core pipeline for **lineament clustering**, developed for the study above.

At the time of publication, we did **not anticipate significant external demand** for the code, and it was left in a **spaghetti-code state**-functional, but difficult to maintain or reuse. However, following unexpected interest and requests for use, we have prepared this **refactored public version**.  This version of the code was designed to improve readability, modularity, and reproducibility from the outset.

**This notebook is also compatible with [Google Colab](https://colab.research.google.com/)**.  
> If you prefer to run the workflow in a browser environment (without any local setup), you can upload the notebook and shapefile to Google Drive and follow the in-notebook instructions in **STEP 1: Specify input shapefile path** to mount your Drive and set file paths.  
> Environment setup (e.g., library installation) is automated in Colab for your convenience.  
> **Before use, please make sure to review your organization’s cloud data policy before handling sensitive files.**

## Erratum Notice

This released code accompanies an **erratum** to the original article, reflecting the improved availability and usability of the method. We hope this version facilitates further use in morphostructural and tectonic studies.


---

## Key Features

- Allows **objective clustering** of lineaments that were originally drawn subjectively by humans.
- Accepts input as **.shp (shapefile)** format and outputs clustered lineaments as separate shapefiles, making it easy to use in GIS workflows.
- Applicable to both **large-scale surveys** (e.g., satellite, ship-based) and **small-scale observations** (e.g., AUV missions).
- Uses a two-step approach for orientation analysis:  
  linearity is first assessed using R² from linear regression, and precise orientation is then extracted using principal component analysis (PCA). This ensures both robust filtering and stable angle estimation.


---
## How to Use

This notebook is designed to be executed step by step from top to bottom.  
Please follow the instructions below:

**1. Set parameters**  
In the cell titled `PARAMETER SETUP`, specify:

- `shapefile_path`: Path to your input `.shp` file  
- `k`: Number of clusters for K-means  
- `R2`: R² threshold for filtering  


**2. Run all cells**  
Once the parameters are set, you can run the remaining cells sequentially.


>**Note For Google Colab users**  
>Please read the comment section in `PARAMETER SETUP` to mount Google Drive and set the file path.
  
> **Note for Image-Based Applications**  
>This tool might be useful for working with lineaments traced from images such as outcrop photographs, cliff-face photos, thin-section micrographs, or even images from deformation experiments (e.g., uniaxial compression tests). 
For example, if you georeference a simple photo in QGIS and export the traced features as a shapefile using a local flat coordinate system, the tool can be applied.
In such cases, using a pseudo-UTM CRS (e.g., EPSG:32600) or a suitable local projection might help preserve linearity without introducing geographic distortion.

---

## Major Updates in Public Release (v1.0.0)

This is the first public release of the tool, refactored from a private version used during the original study. Key improvements include:

- **Geometry parsing**: `extract_lon_lat()` now supports both `LineString` and `MultiLineString`, and skips invalid geometries without interrupting the process.
- **Error transparency**: Detailed logging of `GeoDataFrame` properties when errors occur.
- **Cluster ID logic**: Cluster numbers are now consistently assigned starting from 1, ensuring fixed color mapping between runs.
- **Histogram output**: Added automatic histogram generation for cluster-wise visualization.
- **Slope calculation fix**: A critical bug causing division by zero (especially along east-west boundaries) was corrected.
  > **Note**: Some minor numerical differences were observed before and after the slope calculation fix, mainly due to changes in sorting and numerical precision. The differences in cluster centers were within approximately 0.1–0.2 degrees. These are well within acceptable tolerance and do not affect the clustering structure or interpretation. **This version (v1.0.0) is considered stable and consistent for scientific use.** 

- **Google Colab support**: The tool is now runnable in Google Colab with automated dependency installation and Drive integration.
  > **Note**: When using this tool on Google Colab, input and output files will be accessed via Google Drive.  
  > Please consult your organization's data security policies before uploading or processing any sensitive or internal files on a cloud-based environment.

---

## Change Log

### **v1.0.2: Implementation of Robust Circular Clustering Models**

This update introduces a hybrid clustering framework, enabling a choice between three models. We have transitioned from a simple 1D approximation to more statistically robust methods that respect the periodic nature of directional data.

#### **Supported Models (`clustering_model`)**

* **`original` (1D Cosine Approximation)**
    * The legacy method using a 1-dimensional cosine mapping. 
    * *Note:* Maintained for backward compatibility, but susceptible to "mirroring" issues between NE and NW trends.

* **`2Dkmeans` (2nd Harmonic Mapping / Unit Circle)**
    * **Robustness:** High. Maps angles into a 2D sine-cosine coordinate system (unit circle) via 2nd harmonic (2$\theta$) transformation.
    * **Key Advantage:** Correctly resolves the $0^\circ/180^\circ$ discontinuity and prevents the misclassification of conjugate sets (e.g., NE vs. NW).

* **`vMMM` (von Mises Mixture Model)**
    * **Robustness:** Highest (Statistical Rigor). A probabilistic model using the **von Mises distribution**—the circular equivalent of the Normal distribution.
    * **Advanced Metrics:** Employs an Expectation-Maximization (EM) algorithm to estimate the **Mean Direction ($\mu$)** and the **Concentration Parameter ($\kappa$)**.
    * **Geological Insight:** Provides quantitative measures of how "focused" or "clustered" a specific lineament trend is within the stress field.

---
## Acknowledgements

We sincerely appreciate the exceptional support of **Kea Giles**, editor at *Geology*, whose kind guidance and encouragement were instrumental throughout the publication and erratum process.  
Her thoughtful communication made the experience both constructive and rewarding.

We also gratefully acknowledge the efforts of the entire editorial team at *Geology* for their professional handling of both the manuscript and the accompanying code release.



---
## Output Files Overview

All output files follow the naming pattern: `[basename]`
They are saved in a dedicated output folder automatically created in the same directory as this notebook.


#### Main Outputs

- `00_All_lineament_[basename].png`  
  → Overview plot of all input lineaments.

- `01_k-means_interactive-map_[basename].html`  
  → Interactive map of clustering results (viewable in browser).

- `02_k-means_Result_[basename]_KDE.txt` **†**  
  → Text summary of cluster centers (from KDE analysis).

#### Histograms & Trends

- `03_k-means_Trend_[basename]_Cluster_*.png`  
  → Histograms of trend angles per cluster.

- `03_k-means_Trend_[basename]_hist.png`  
  → Combined histogram of all clusters.

- `04_k-means_Trend_[basename].png`  
  → KDE-based trend plot (corresponds to 02).


#### Rose Diagrams

- `05_k-means_rosem_[basename].(png/eps)` **†**  
  → Rose diagram of clustered lineaments.

- `05_k-means_rosem_compare_[basename].(png/eps)`  
  → Combined rose diagram for all clusters.

#### Clustering Evaluation

- `06_elbow_[basename].png`  
  → Elbow plot to help select number of clusters (k).

#### Clustered Data Exports

- `07_cluster_[n]_[basename].csv`  
  → Clustered lineament data (per cluster, CSV format).

- `exported_shapefiles/cluster_[n]_[basename].{shp,shx,dbf,prj,cpg}` **†**  
  → Shapefile set for each cluster (compatible with QGIS/ArcGIS).

**† = Used in the published paper.**

---

## License

This software is released under the **MIT License**.

**Copyright (c) 2025 Hiroaki Koge, Hiroki Minami, Yumiko Harigane**

Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the “Software”), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED “AS IS”, WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.

This software uses the following third-party Python packages, each distributed under a license compatible with the MIT License: NumPy, Pandas, Matplotlib, GeoPandas, Folium, Shapely, PyProj, Geopy, PyGMT, SciPy, scikit-learn. Users are advised to consult each package's documentation for individual licensing details.


In [ ]:
# =============================================
# PARAMETER SETUP (Please edit this section only)
# =============================================
'''
Shapefile requirements:
Ensure the following files are located in the same folder:

- .shp : Geometry data  
- .shx : Geometry index  
- .dbf : Attribute table  
- .prj : Coordinate Reference System (CRS)  
         Required for accurate reprojection. If missing, EPSG:32652 (UTM Zone 52N for Japan) will be assumed.  
- (Optional) .cpg, .sbn, .sbx
'''

shapefile_path = "./your_data.shp"  # ← Replace with your shapefile path
k = 3                                # Number of clusters for K-means
R2 = 0.95                            # R² threshold for filtering

# Options: 'original' (1D cos), '2Dkmeans' (sin-cos mapping), 'vMMM' (von Mises Mixture)
clustering_model = 'original'

# =============================================
# Google Colab Setup (if needed)
# Uncomment the block below if running in Google Colab
# =============================================

# try:
#     import google.colab
#     from google.colab import drive
#     drive.mount('/content/drive')
#     shapefile_path = "/content/drive/MyDrive/path_to_your_data/your_data.shp"
# except ImportError:
#     pass


In [ ]:
# If running on Google Colab, install missing packages
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !sudo apt-get install -y gmt gmt-common libgmt-dev
    !pip install geopandas folium shapely pyproj geopy pygmt scikit-learn --quiet matplotlib

# === Standard library imports ===
import os
import sys
import math
import random
import pickle
from datetime import datetime
from math import atan2, degrees  # if using atan2, degrees directly

# === Core third-party imports ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

# === Geospatial & Mapping ===
import folium
from folium.plugins import FeatureGroupSubGroup, PolyLineTextPath
import pygmt
from shapely.geometry import LineString, MultiLineString
from pyproj import Geod
from geopy.distance import geodesic

# === Statistics ===
from scipy.stats import linregress
from sklearn.cluster import KMeans
from sklearn.neighbors import KernelDensity
from sklearn.decomposition import PCA

In [ ]:
# Generate output folder based on shapefile name and date
# Extract the base name from the shapefile path
filename = os.path.basename(shapefile_path)
basename = os.path.splitext(filename)[0]  # e.g., "OT_lineament_wgs84_worldmercator_COT"

# Generate today's date string
today = datetime.now().strftime("%Y%m%d")

# Determine the directory containing the shapefile
parent_dir = os.path.dirname(os.path.abspath(shapefile_path))

# Construct the full path to the output directory
output_dir = os.path.join(parent_dir, f"{basename}_{today}")

# Create the folder if it does not already exist
os.makedirs(output_dir, exist_ok=True)

print(f"[INFO] Output will be saved to: {output_dir}")

gdf = gpd.read_file(shapefile_path)

In [ ]:
def check_shapefile_components(shapefile_path):
    # Get the directory and base filename (without extension) of the specified shapefile
    directory = os.path.dirname(shapefile_path)
    base_filename = os.path.splitext(os.path.basename(shapefile_path))[0]

    # Construct paths for associated files
    cpg_path = os.path.join(directory, f"{base_filename}.cpg")
    dbf_path = os.path.join(directory, f"{base_filename}.dbf")
    sbn_path = os.path.join(directory, f"{base_filename}.sbn")
    sbx_path = os.path.join(directory, f"{base_filename}.sbx")
    shx_path = os.path.join(directory, f"{base_filename}.shx")
    prj_path = os.path.join(directory, f"{base_filename}.prj")  # Path to the .prj file

    # Read the .shp file and check the CRS (Coordinate Reference System)
    try:
        gdf = gpd.read_file(shapefile_path)
        if gdf.crs is None:
            print("Shapefile CRS: None (No CRS information found in .shp or .prj file)")
            # Check CRS information in the .prj file if it exists
            if os.path.exists(prj_path):
                with open(prj_path, 'r') as prj_file:
                    prj_content = prj_file.read().strip()
                    print(f".prj file contents: {prj_content}")
            else:
                print(".prj file not found.")
        else:
            print(f"Shapefile CRS: {gdf.crs}")
    except Exception as e:
        print(f"Error reading {shapefile_path}: {e}")

    # Read the .cpg file for encoding information
    if os.path.exists(cpg_path):
        try:
            with open(cpg_path, 'r') as file:
                encoding = file.read().strip()
                print(f".cpg file encoding: {encoding}")
        except Exception as e:
            print(f"Error reading {cpg_path}: {e}")
    else:
        print(".cpg file not found.")

    # Read the .dbf file for attribute data
    if os.path.exists(dbf_path):
        try:
            df = pd.read_csv(dbf_path, encoding='latin1')  # Specify encoding
            print(".dbf file contents (first 5 rows):")
            print(df.head())
        except Exception as e:
            print(f"Error reading {dbf_path}: {e}")
    else:
        print(".dbf file not found.")

    # Check for .sbn and .sbx files (spatial index files)
    if os.path.exists(sbn_path) and os.path.exists(sbx_path):
        print(".sbn and .sbx files exist (spatial index files).")
    else:
        print(".sbn or .sbx file not found.")

    # Check for .shx file (index file)
    if os.path.exists(shx_path):
        print(".shx file exists (index file).")
    else:
        print(".shx file not found.")

# Call the function to check the shapefile components
check_shapefile_components(shapefile_path)  # Replace with the path to your shapefile


In [ ]:
def extract_lon_lat(geom):
    try:
        if isinstance(geom, LineString):
            coords = list(geom.coords)
            return pd.DataFrame(coords, columns=['Lon', 'Lat'])
        elif isinstance(geom, MultiLineString):
            dfs = []
            for part in geom.geoms:
                coords = list(part.coords)
                dfs.append(pd.DataFrame(coords, columns=['Lon', 'Lat']))
            return pd.concat(dfs, ignore_index=True)
        else:
            raise TypeError(f"Unsupported geometry type: {type(geom)}")
    except Exception as e:
        print(f"Warning: Skipped invalid geometry at index due to error: {e}")
        return pd.DataFrame(columns=['Lon', 'Lat'])

# Extract coordinates for each geometry and store them in a list
extracted_data = []
id_counter = 0  # Initialize the ID counter

for idx, row in gdf.iterrows():
    line_df = extract_lon_lat(row['geometry'])  # Extract coordinates
    line_df['id'] = id_counter  # Assign an ID
    extracted_data.append(line_df)
    id_counter += 1

# Combine all data into a single DataFrame
dff = pd.concat(extracted_data, ignore_index=True)

# Check the result
print(dff.head())

In [ ]:
def display_geodataframe_info(gdf):
    # Display basic information about the GeoDataFrame
    print("GeoDataFrame Information:")
    print(gdf.info())

    # Display the Coordinate Reference System (CRS)
    print("\nCoordinate Reference System (CRS):")
    print(gdf.crs)

    # Display the head (first 5 rows) of the GeoDataFrame
    print("\nHead of the GeoDataFrame:")
    print(gdf.head())

    # Display all column names
    print("\nColumn Names:")
    print(gdf.columns)

    # Display geometry information (first 5 geometries)
    print("\nGeometry (first 5 geometries):")
    print(gdf.geometry.head())

    # Display the bounding box (overall extent of the geometries)
    print("\nBounding Box:")
    print(gdf.total_bounds)

    # Display the data types of each column
    print("\nColumn Data Types:")
    print(gdf.dtypes)

    # Display descriptive statistics for the GeoDataFrame
    print("\nDescriptive Statistics:")
    print(gdf.describe())

    # Display descriptive statistics for the GeoDataFrame
    print("\nLinStfingType:")
    print(gdf.geom_type.value_counts())

# Call the function to display GeoDataFrame details
display_geodataframe_info(gdf)

In [ ]:
# Validate geometry in the shapefile

# Count how many geometries are empty
print("Number of empty geometries:", gdf.is_empty.sum())

# Count how many geometries are missing (NaN)
print("Number of missing geometries:", gdf['geometry'].isna().sum())

# Check if each geometry can be accessed via .xy (LineString or similar)
for i, geom in enumerate(gdf['geometry']):
    try:
        _ = geom.xy  # Test if .xy can be accessed
    except NotImplementedError:
        print(f"Row {i}: NotImplementedError;geometry type not supported for .xy")
    except Exception as e:
        print(f"Row {i}: Other error: {e}")

# Optional: Inspect a specific geometry manually if needed
# print(gdf.loc[3559, 'geometry'])

In [ ]:
def convert_to_wgs84(gdf):
    # If the current CRS is not set, assume UTM 52R (EPSG:32652) for Japan region
    if gdf.crs is None:
        print(" Warning: No CRS found in shapefile.")
        print(" Assuming UTM Zone 52N (EPSG:32652), valid for Japan region (e.g., Tokyo).")
        gdf = gdf.set_crs("EPSG:32652")  # UTM 52N for Japan (Northern Hemisphere)

    # Display the current CRS
    print("Current CRS:", gdf.crs)

    # Convert to WGS84
    gdf = gdf.to_crs("EPSG:4326")

    # Display the CRS after conversion
    print("Converted CRS:", gdf.crs)

    return gdf


    # Display the current CRS
    print("Current CRS:", gdf.crs)

    # Convert to WGS84
    gdf = gdf.to_crs("EPSG:4326")

    # Display the CRS after conversion
    print("Converted CRS:", gdf.crs)

    return gdf

# Convert from UTM 52R to WGS84 (Northern Hemisphere)
gdf_converted = convert_to_wgs84(gdf)

# Display the converted GeoDataFrame
print(gdf_converted.head())

In [ ]:
# Extract coordinates from each LINESTRING
extracted_data = []
id_counter = 0  # Initialize the ID counter

for idx, row in gdf_converted.iterrows():
    line_df = extract_lon_lat(row['geometry'])  # Assumes extract_lon_lat is defined elsewhere
    line_df['id'] = id_counter  # Assign a unique ID to each line
    extracted_data.append(line_df)
    id_counter += 1  # Increment the ID counter

# Combine all extracted coordinate data into a single DataFrame
dff = pd.concat(extracted_data, ignore_index=True)

# Reconstruct LINESTRINGs from the extracted points (suppress deprecation warning)
lines = (
    dff.groupby('id')[['Lon', 'Lat']]
    .apply(lambda x: LineString(zip(x['Lon'], x['Lat'])))
)

# Create a GeoDataFrame with reconstructed LineStrings
gdf_reconstructed = gpd.GeoDataFrame({'id': lines.index, 'geometry': lines}, crs="EPSG:4326")



# Plot all reconstructed lines, colored by ID
fig, ax = plt.subplots(figsize=(10, 10))

# Plot the reconstructed geometries with different colors per ID
gdf_reconstructed.plot(ax=ax, column='id', linewidth=2, legend=True)

# Configure plot appearance
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Index Map with Original SHP")
plt.grid(True)

# Save the figure to the output directory
out0 = os.path.join(output_dir, f"00_All_lineament_{basename}.png")
plt.savefig(out0)
plt.show()

In [ ]:
# Compute robust regression statistics for each line segment
dg = pd.DataFrame()  # Initialize an empty DataFrame to store the results

for id in dff["id"].unique():
    # Extract x (longitude) and y (latitude) values for the current line
    x = dff.loc[dff["id"] == id, "Lon"].values
    y = dff.loc[dff["id"] == id, "Lat"].values

    # Check if there are at least two points to compute regression
    if len(x) < 2:
        print(f"Skipping ID {id};not enough points for regression.")
        continue

    try:
        result = linregress(x, y)

        # Append results to the output DataFrame
        dg = pd.concat([dg, pd.DataFrame({
            "id": [id],
            "correlation_coefficient": [result.rvalue],
            "slope": [result.slope],
            "intercept": [result.intercept]
        })], ignore_index=True)

    except Exception as e:
        print(f"Error processing ID {id}: {e}")
        continue

# Show the result DataFrame
dg

In [ ]:
def remove_curved_line(dataframe, df):
    """
    Remove line segments with low linearity based on correlation coefficient.
    Parameters:
    - dataframe: DataFrame containing correlation statistics (e.g., from linregress)
    - df: Original DataFrame containing point data (with 'id' column)
    Returns:
    - filtered_data: DataFrame with only well-aligned (linear) segments
    """

    # Filter: keep only rows where the absolute correlation coefficient exceeds the threshold
    filtered_df = dataframe[dataframe['correlation_coefficient'].abs() > R2]

    # Identify any duplicated 'id' values (just in case)
    duplicated_rows = filtered_df[filtered_df['id'].duplicated()]

    # Remove duplicated rows (if any)
    filtered_df = filtered_df.drop(duplicated_rows.index)

    # Count the number of unique valid line IDs
    i_id_count = filtered_df['id'].nunique()

    # Extract the list of valid IDs
    unique_i_ids = filtered_df['id'].unique()

    # From the full data, keep only rows whose 'id' is in the valid list
    filtered_data = df[df['id'].isin(unique_i_ids)]

    return filtered_data


In [ ]:
filtered_df=remove_curved_line(dg,dff)

unique_values1 = dff['id'].unique()
unique_values2 = filtered_df['id'].unique()

print('from',len(unique_values1), 'lines >',len(unique_values2),' lines')

In [ ]:
def calculate_bearing(start_lon, start_lat,  end_lon,end_lat):
    azimuth = atan2(end_lon - start_lon, end_lat - start_lat)
    azimuth = degrees(azimuth)
    if azimuth < 0:
        azimuth += 360
    return azimuth

def calculate_distance(start_lon, start_lat, end_lon,end_lat):
    start_point = (start_lat, start_lon)
    end_point = (end_lat, end_lon)
    distance = geodesic(start_point, end_point).kilometers
    return distance

def calculate_pca_bearing(lons, lats):
    coords = np.column_stack((lons, lats))  # shape (N, 2)
    pca = PCA(n_components=2)
    pca.fit(coords)

    # direction 
    vec = pca.components_[0]
    vx, vy = vec

    # convert map scale
    theta_rad = atan2(vx, vy) 
    theta_deg = degrees(theta_rad)

    bearing = (theta_deg + 360) % 360
    return bearing

In [ ]:
filtered_df = filtered_df.copy()

for i_id, group in filtered_df.groupby('id'):
    lons = group['Lon'].values
    lats = group['Lat'].values

    direction = calculate_pca_bearing(lons, lats)
    filtered_df.loc[group.index, 'direction'] = direction
    #
    
    distance = geodesic((lats[0], lons[0]), (lats[-1], lons[-1])).kilometers
    filtered_df.loc[group.index, 'distance'] = distance
    
filtered_df

In [ ]:
# make dataframe [i_id, direction, distance]
result_df = filtered_df.groupby('id')[['direction', 'distance']].first().reset_index()

# unique i_id
dg_azimdis = result_df.drop_duplicates(subset='id').reset_index(drop=True)

In [ ]:
# direction : 0-180
direction_deg = dg_azimdis['direction'].values.copy()
direction_deg[direction_deg > 180] -= 180
dg_azimdis['direction2'] = direction_deg

In [ ]:
# =============================================
# Helper: Von Mises Mixture Model Class
# =============================================
from scipy.stats import vonmises
from sklearn.base import BaseEstimator, ClusterMixin

class VonMisesMixture(BaseEstimator, ClusterMixin):
    def __init__(self, n_clusters=3, max_iter=100, tol=1e-4):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
    def fit(self, X):
        n_samples = X.shape[0]
        self.mus = np.linspace(0, 2 * np.pi, self.n_clusters, endpoint=False)
        self.kappas = np.ones(self.n_clusters) * 5.0
        self.weights = np.ones(self.n_clusters) / self.n_clusters
        for _ in range(self.max_iter):
            probs = np.zeros((n_samples, self.n_clusters))
            for k in range(self.n_clusters):
                probs[:, k] = self.weights[k] * vonmises.pdf(X, self.kappas[k], loc=self.mus[k])
            sum_probs = probs.sum(axis=1, keepdims=True)
            sum_probs[sum_probs == 0] = 1e-15
            responsibilities = probs / sum_probs
            prev_mus = self.mus.copy()
            for k in range(self.n_clusters):
                resp_k = responsibilities[:, k]
                N_k = resp_k.sum()
                if N_k < 1e-5: continue
                s, c = (resp_k * np.sin(X)).sum(), (resp_k * np.cos(X)).sum()
                self.mus[k] = np.arctan2(s, c)
                r_bar = np.sqrt(s**2 + c**2) / N_k
                if r_bar < 0.53: self.kappas[k] = 2 * r_bar + r_bar**3 + (5 * r_bar**5) / 6
                elif r_bar < 0.85: self.kappas[k] = -0.4 + 1.39 * r_bar + 0.43 / (1 - r_bar)
                else: self.kappas[k] = 1 / (r_bar**3 - 4 * r_bar**2 + 3 * r_bar)
                self.weights[k] = N_k / n_samples
            if np.allclose(self.mus, prev_mus, rtol=self.tol): break
        return self
    def predict(self, X):
        probs = np.zeros((X.shape[0], self.n_clusters))
        for k in range(self.n_clusters):
            probs[:, k] = self.weights[k] * vonmises.pdf(X, self.kappas[k], loc=self.mus[k])
        return np.argmax(probs, axis=1)


In [ ]:
# --- Pre-processing ---
direction_deg = dg_azimdis['direction'].values.copy()
direction_deg[direction_deg > 180] -= 180
dg_azimdis['direction2'] = direction_deg
theta_rad = np.radians(dg_azimdis['direction2'].values)

# --- Feature Preparation (エルボー法のためにここで定義しますわ！) ---
# 1D空間用
features_1d = np.cos(theta_rad).reshape(-1, 1)
# 2D空間用 (2-theta mapping)
features_2d = np.column_stack([np.cos(2 * theta_rad), np.sin(2 * theta_rad)])

# --- Clustering Execution ---
if clustering_model == 'original':
    print("[INFO] Running original 1D-cosine clustering...")
    features = features_1d
    model = KMeans(n_clusters=k, random_state=0, n_init=10)
    initial_labels = model.fit_predict(features)

elif clustering_model == '2Dkmeans':
    print("[INFO] Running 2nd-harmonic 2D (sin-cos) clustering...")
    features = features_2d
    model = KMeans(n_clusters=k, random_state=0, n_init=10)
    initial_labels = model.fit_predict(features)

elif clustering_model == 'vMMM':
    print("[INFO] Running strict von Mises Mixture Model (vMMM)...")
    # vMMMは確率モデルですが、エルボー法(KMeans)の比較用に2D特徴量をセットしておきますわ
    features = features_2d 
    phi_rad = 2 * theta_rad
    vmmm = VonMisesMixture(n_clusters=k)
    initial_labels = vmmm.fit(phi_rad).predict(phi_rad)
    for i in range(k):
        mu_deg = np.degrees(vmmm.mus[i]/2) % 180
        print(f"  Cluster {i}: Mean Direction = {mu_deg:.1f}°, kappa = {vmmm.kappas[i]:.2f}")

# --- Label Sorting ---
df_tmp = pd.DataFrame({'label': initial_labels})
label_order = (df_tmp.value_counts('label').reset_index(name='count')
               .sort_values('count', ascending=False).reset_index(drop=True))
label_map = {old: new + 1 for new, old in enumerate(label_order['label'])}
dg_azimdis['cluster'] = [label_map[l] for l in initial_labels]

print(f"\n[DONE] Clustering complete using '{clustering_model}'. 'features' is ready for elbow method!")

In [ ]:
# cluster sample counts:
cluster_counts = dg_azimdis['cluster'].value_counts()

print("Cluster sample counts:")
print(cluster_counts)

In [ ]:
filtered_df = filtered_df.copy()

# split filtered_df in each clusters
filtered_df.loc[:, 'cluster'] = filtered_df['id'].map(dg_azimdis.set_index('id')['cluster'])

dh = [] 
for i in range(1, k + 1):
    dhi = filtered_df.loc[filtered_df['cluster'] == i].copy()
    dh.append(dhi)

In [ ]:
# List to store WCSS values
wcss = []

# Range of number of clusters (for this example, we're trying from 1 to 10)
max_clusters = 10
for i in range(1, max_clusters + 1):
    kmeans = KMeans(n_clusters=i, init='k-means++', max_iter=300, n_init=10, random_state=0)
    kmeans.fit(features)
    wcss.append(kmeans.inertia_)

# Plotting the elbow graph
plt.figure(figsize=(10,5))
plt.plot(range(1, max_clusters + 1), wcss, marker='o', linestyle='--')
plt.title('Elbow Method for Optimal Number of Clusters')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')

# 
out2 = os.path.join(output_dir, f"06_elbow_{basename}.png")
plt.savefig(out2)
plt.show()

In [ ]:
for i, sub_df in enumerate(dh, start=1): 
    filename = os.path.join(output_dir, f'07_cluster_{i}_{basename}.csv')
    sub_df.to_csv(filename, index=False)

In [ ]:
with open('dataframes_list.pkl', 'wb') as f:
    pickle.dump(dh, f)

In [ ]:
result_df

In [ ]:
# calc center of the data
center_lat = dff['Lat'].mean()
center_lon = dff['Lon'].mean()

# initialize map set
map = folium.Map(location=[center_lat, center_lon], zoom_start=12, tiles='OpenStreetMap')

colors = ['#FF0000','#0000FF', '#FFA500', '#00FF00',  '#FFFF00',  '#FF00FF', '#00FFFF']

# 
for i, df_cluster in enumerate(dh):
    cluster_color = colors[i]
    for i_id, group in df_cluster.groupby('id'):
        points = list(zip(group['Lat'], group['Lon']))
        folium.PolyLine(points, color=cluster_color, weight=2, opacity=0.7).add_to(map)

out1 = os.path.join(output_dir, f'01_k-means_interactive-map{basename}.html')
map.save(out1)

map

In [ ]:
# Create a subdirectory for exported shapefiles
shapefile_dir = os.path.join(output_dir, "exported_shapefiles")
os.makedirs(shapefile_dir, exist_ok=True)

# Loop through each cluster and export as a shapefile
for i, df_cluster in enumerate(dh):
    cluster_color = colors[i]
    lines = []

    for id, group in df_cluster.groupby('id'):
        points = list(zip(group['Lon'], group['Lat']))  # Format: (Lon, Lat)
        line = LineString(points)
        lines.append({'geometry': line, 'color': cluster_color, 'id': id})

    gdf = gpd.GeoDataFrame(lines, crs="EPSG:4326")

    # Define the output shapefile path
    out_shapefile = os.path.join(shapefile_dir, f'cluster_{i + 1}_{basename}.shp')
    gdf.to_file(out_shapefile)

    print(f"Shapefile saved to: {out_shapefile}")


In [ ]:
# # Plot direction histograms for each cluster as side-by-side subplots

# fig, axes = plt.subplots(1, k, figsize=(12, 4))  # Create 1 row with k subplots

# # Get sorted list of unique cluster IDs
# clusters = dg_azimdis['cluster'].unique()
# clusters.sort()

# # Plot histogram for each cluster
# for i, cluster in enumerate(clusters):
#     cluster_data = dg_azimdis[dg_azimdis['cluster'] == cluster]
#     axes[i].hist(cluster_data['direction'], bins=30, color=colors[i], alpha=0.7)
#     axes[i].set_title(f'Cluster {cluster}')
#     axes[i].set_xlabel('Direction (°)')
#     axes[i].set_ylabel('Frequency')

# plt.tight_layout()

# # Save the figure to file
# out2 = os.path.join(output_dir, f'02_k-means_histogram_{basename}.png')
# plt.savefig(out2)

# plt.show()


In [ ]:
# Kernel Density Estimation (KDE) for each cluster

# KDE parameter
bandwidth = 0.5

# Prepare variables to store peak density info
k_values = range(1, k + 1)
max_densities = []
max_x_values = []

# Start a new figure
plt.figure()

# Loop through each cluster
for i in range(1, k + 1):
    cluster_data = dg_azimdis[dg_azimdis['cluster'] == i]
    data_array = np.array(cluster_data['direction2'])

    # Perform KDE with Gaussian kernel
    kde = KernelDensity(kernel='gaussian', bandwidth=bandwidth).fit(data_array[:, np.newaxis])
    x = np.linspace(data_array.min(), data_array.max(), 100)
    log_density = kde.score_samples(x[:, np.newaxis])

    # Plot the density curve
    plt.plot(x, np.exp(log_density), color=colors[i - 1])

    # Identify peak density and corresponding x value
    max_density_index = np.argmax(log_density)
    max_density = np.exp(log_density[max_density_index])
    max_x = x[max_density_index]

    # Store results
    max_densities.append(max_density)
    max_x_values.append(max_x)

# Plot styling
plt.xlabel('Direction (°)')
plt.ylabel('Density')
plt.title('Kernel Density Estimation (KDE) by Cluster')
legend_labels = [f'Cluster {k_value}' for k_value in k_values]
plt.legend(legend_labels)

# Save the plot as an image
out_img = os.path.join(output_dir, f"04_k-means_Trend_{basename}.png")
plt.savefig(out_img)
print(f"KDE plot saved (normalized histograms overlaid): {out_img}")

plt.show()

# Save peak density results to a text file

out_txt = os.path.join(output_dir, f"02_k-means_Result_{basename}_KDE.txt")
with open(out_txt, 'w') as f:
    for k_value, max_density, max_x in zip(k_values, max_densities, max_x_values):
        result = f"Cluster {k_value}: Max Density = {max_density:.3f} at X = {max_x:.3f}\n"
        f.write(result)
        print(result.strip())

    f.write("\nCluster sample counts:\n")
    cluster_counts_str = cluster_counts.to_string()
    f.write(cluster_counts_str)

print(f"Text results saved to: {out_txt}")


In [ ]:
# Plot normalized histograms for each cluster with common bins

# Create common bin edges across all data (e.g., 30 bins)
bins = np.linspace(dg_azimdis['direction2'].min(), dg_azimdis['direction2'].max(), 30)

plt.figure()  # Start a new figure

# Plot histograms for each cluster
for i in range(1, k + 1):
    cluster_data = dg_azimdis[dg_azimdis['cluster'] == i]

    # Use density=True to normalize, and alpha to control transparency
    plt.hist(cluster_data['direction2'], bins=bins, density=True, alpha=0.5,
             color=colors[i - 1], label=f'Cluster {i}')

# Set axis labels and title
plt.xlabel('Direction (°)')
plt.ylabel('Normalized Frequency')
plt.title('Normalized Histograms for Each Cluster')
plt.legend()

# Save the plot as an image
out_img = os.path.join(output_dir, f"03_k-means_Trend_{basename}_hist.png")
plt.savefig(out_img)
print(f"Histogram image saved to: {out_img}")

plt.show()


In [ ]:
# Determine common axis limits for all histograms

# Use predefined `bins`;x-axis will range from bins[0] to bins[-1]

# Compute the maximum y-axis value across all clusters (normalized histograms)
global_max_y = 0
for i in range(1, k + 1):
    cluster_data = dg_azimdis[dg_azimdis['cluster'] == i]['direction2']
    hist, _ = np.histogram(cluster_data, bins=bins, density=True)
    global_max_y = max(global_max_y, hist.max())

# Add a margin (e.g., 10%) to the maximum y for visual clarity
global_max_y *= 1.1


# Save individual histograms per cluster

for i in range(1, k + 1):
    plt.figure()
    cluster_data = dg_azimdis[dg_azimdis['cluster'] == i]

    # Plot normalized histogram using the shared bin edges
    plt.hist(cluster_data['direction2'], bins=bins, density=True, alpha=1,
             color=colors[i - 1])

    # Axis labels and title
    plt.xlabel('Direction (°)')
    plt.ylabel('Normalized Frequency')
    plt.title(f'Normalized Histogram for Cluster {i}')

    # Apply shared axis limits
    plt.xlim(bins[0], bins[-1])
    plt.ylim(0, global_max_y)

    # Save the figure
    out_img_indiv = os.path.join(output_dir, f"03_k-means_Trend_{basename}_Cluster_{i}_hist.png")
    plt.savefig(out_img_indiv)
    print(f"Cluster {i} histogram saved: {out_img_indiv}")

    plt.show()


In [ ]:
# Prepare cluster data for rose diagram (assumes 1-based cluster labels)

# Extract direction values per cluster
cluster_data = [
    dg_azimdis[dg_azimdis['cluster'] == label]['direction']
    for label in range(1, k + 1)
]

v = []            # List to hold raw direction values
hist_data = []    # List to hold histogram DataFrames for each cluster

# Compute histograms for each cluster
for i in range(len(cluster_data)):
    if len(cluster_data[i]) > 0:
        v.append(cluster_data[i].values)
        frequencies, bins = np.histogram(v[i], bins=30, range=(0, 180))
        directions = [(bins[j] + bins[j + 1]) / 2 for j in range(len(bins) - 1)]
        hist_data.append(pd.DataFrame({'Direction': directions, 'Frequency': frequencies}))
    else:
        hist_data.append(pd.DataFrame({'Direction': [], 'Frequency': []}))


# Generate rose diagrams for each cluster using PyGMT


fig = pygmt.Figure()

with fig.subplot(
    nrows=1, ncols=k, figsize=("16c", "9c"), margins="8c"
):
    for i in range(1, k + 1):
        df = hist_data[i - 1]
        if not df.empty:
            max_frequency_row = df.loc[df['Frequency'].idxmax()]
            max_frequency_direction = max_frequency_row['Direction']
            with fig.set_panel(panel=i - 1):
                fig.rose(
                    length=df['Frequency'],
                    azimuth=df['Direction'],
                    region=[0, 1, 0, 360],  # polar coordinates
                    diameter="7.5c",
                    sector="6+r",          # 6° sectors, relative mode
                    norm=True,             # normalize to percentage
                    fill=colors[i - 1],
                    frame=["x0.2g0.2", "y15g15", "+gwhite"],
                    pen="1p"
                )


# Save the figure


out3_eps = os.path.join(output_dir, f'05_k-means_rosem_{basename}.eps')
out3_png = os.path.join(output_dir, f'05_k-means_rosem_{basename}.png')
fig.savefig(out3_eps)
fig.savefig(out3_png)

fig.show()


In [ ]:
# Generate two rose diagrams for comparison (raw vs normalized)

# Set common fill color for all sectors
all_color = '#808080'

# Output file paths
out_compare_eps = os.path.join(output_dir, f'05_k-means_rosem_compare_{basename}.eps')
out_compare_png = os.path.join(output_dir, f'05_k-means_rosem_compare_{basename}.png')


# Prepare raw direction data (all data combined)
all_direction_data = dg_azimdis['direction'].values
frequencies_all, bins_all = np.histogram(all_direction_data, bins=30, range=(0, 180))
directions_all = [(bins_all[j] + bins_all[j + 1]) / 2 for j in range(len(bins_all) - 1)]

df_raw_all = pd.DataFrame({
    'Direction': directions_all,
    'Frequency': frequencies_all
})

# Prepare normalized histogram by summing all clusters
sum_frequency = np.zeros_like(frequencies_all, dtype=float)

for df in hist_data:
    if not df.empty:
        freq_norm = df['Frequency'] / df['Frequency'].sum()  # Normalize to proportions
        sum_frequency += freq_norm.values

df_norm_all = pd.DataFrame({
    'Direction': directions_all,
    'Frequency': sum_frequency
})

# Plot side-by-side rose diagrams
fig = pygmt.Figure()

with fig.subplot(nrows=1, ncols=2, figsize=("20c", "10c"), margins="1c"):

    # Left panel: raw counts (absolute values)
    with fig.set_panel(0):
        fig.rose(
            length=df_raw_all['Frequency'],
            azimuth=df_raw_all['Direction'],
            region=[0, 1, 0, 360],
            diameter="8c",
            sector="6+r",
            norm=True,  # Normalize for display
            fill=all_color,
            frame=["x0.2g0.2", "y15g15", "+gwhite"],
            pen="1p"
        )

    # Right panel: cluster-normalized values (sum of proportions)
    with fig.set_panel(1):
        fig.rose(
            length=df_norm_all['Frequency'],
            azimuth=df_norm_all['Direction'],
            region=[0, 1, 0, 360],
            diameter="8c",
            sector="6+r",
            norm=True,
            fill=all_color,
            frame=["x0.2g0.2", "y15g15", "+gwhite"],
            pen="1p"
        )

# Display and save
fig.show()
fig.savefig(out_compare_eps)
fig.savefig(out_compare_png)

print(f"Comparison saved: Left = raw counts, Right = normalized cluster proportions\n{out_compare_png}")


In [ ]:
max_frequency_row = hist_data[0].loc[hist_data[0]['Frequency'].idxmax()]
max_frequency_direction = max_frequency_row['Direction']

In [ ]:
with open(out_txt, 'a') as f:
    result_count = f"\n\nFiltered from {max(dff.id)} to {len(dg_azimdis)} lines\n"
    f.write(result_count)

In [ ]:
print('end of program')